# Coding a spam classifier with naive Bayes

### Importing packages

In [1]:
import numpy as np
import pandas as pd

### Loading the dataset
For the next two cells, please run only one of them.
- Run the first cell if you cloned the Github Repo
- Run the second cell if you opened this as a Google Colab

In [2]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU HAVE CLONED THE REPO
emails = pd.read_csv('emails.csv')
emails[:10]

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
5,"Subject: great nnews hello , welcome to medzo...",1
6,Subject: here ' s a hot play in motion homela...,1
7,Subject: save your money buy getting this thin...,1
8,Subject: undeliverable : home based business f...,1
9,Subject: save your money buy getting this thin...,1


In [2]:
# IMPORTANT: ONLY RUN THIS CELL IF YOU ARE WORKING ON A COLAB 

url = "https://raw.githubusercontent.com/luisguiserrano/manning/master/Chapter_08_Naive_Bayes/emails.csv"
emails = pd.read_csv(url)
emails.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


### Pre-processing the email

In [3]:
def process_email(text):
    text = text.lower()
    return list(set(text.split()))

emails['words'] = emails['text'].apply(process_email)

In [4]:
emails[:10]

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[good, stylish, collaboration, even, we, our, ..."
1,Subject: the stock trading gunslinger fanny i...,1,"[edt, chameleon, sapling, stock, hall, chisel,..."
2,Subject: unbelievable new homes made easy im ...,1,"[foward, homes, 72, been, extended, look, unbe..."
3,Subject: 4 color printing special request add...,1,"[click, ca, order, graphix, golden, our, @, ad..."
4,"Subject: do not have money , get software cds ...",1,"[ain, to, ,, comedies, by, ended, software, al..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[worldwide, pleased, phar, s, shakedown, day, ..."
6,Subject: here ' s a hot play in motion homela...,1,"[terms, even, al, vnbl, areas, pubiic, perfect..."
7,Subject: save your money buy getting this thin...,1,"[tried, bed, effect, exactiy, any, country, ev..."
8,Subject: undeliverable : home based business f...,1,"[25, 0, id, :, subject, sent, tfi, of, kpn, fe..."
9,Subject: save your money buy getting this thin...,1,"[tried, bed, effect, exactiy, any, country, ev..."


### Calculating the prior (probability that a random email is spam)

For this, we simply divide the number of spam emails by the total number of emails in the dataset.

In [5]:
num_emails = len(emails)
num_spam = sum(emails['spam'])

print("Number of emails:", num_emails)
print("Number of spam emails:", num_spam)
print()

# Calculating the prior probability that an email is spam
print("Probability of spam:", num_spam/num_emails)

Number of emails: 5728
Number of spam emails: 1368

Probability of spam: 0.2388268156424581


### Training a naive Bayes model

Our plan is to write a dictionary, and in this dictionary record every word, and its pair of occurrences in spam and ham.

In [6]:
model = {}

# Training process
for index, email in emails.iterrows():
    for word in email['words']:
        if word not in model:
            model[word] = {'spam': 1, 'ham': 1}
        if word in model:
            if email['spam']:
                model[word]['spam'] += 1
            else:
                model[word]['ham'] += 1

In [7]:
model['lottery']

{'spam': 9, 'ham': 1}

In [8]:
model['sale']

{'spam': 39, 'ham': 42}

The counts start at 1 (to avoid zero counts), so *lottery* appears in 8 spam emails and 0 ham emails, and *sale* appears in 38 spam emails and 41 ham emails.

### Using the model to make predictions

In [9]:
def predict_bayes(word):
    word = word.lower()
    num_spam_with_word = model[word]['spam']
    num_ham_with_word = model[word]['ham']
    return 1.0*num_spam_with_word/(num_spam_with_word + num_ham_with_word)

In [10]:
predict_bayes('lottery')

0.9

In [11]:
predict_bayes('sale')

0.48148148148148145

In [12]:
def predict_naive_bayes(email):
    total = len(emails)
    num_spam = sum(emails['spam'])
    num_ham = total - num_spam
    email = email.lower()
    words = set(email.split())
    spams = [1.0]
    hams = [1.0]
    for word in words:
        if word in model:
            spams.append(model[word]['spam']/num_spam*total)
            hams.append(model[word]['ham']/num_ham*total)
    prod_spams = float(np.prod(spams)*num_spam)
    prod_hams = float(np.prod(hams)*num_ham)
    return prod_spams/(prod_spams + prod_hams)

Notice that the model predicts high probability for the messages that sound spammy, and low probabilities for those that sound legitimate.

In [13]:
# Spammy
predict_naive_bayes('lottery sale')

0.9638144470140118

In [14]:
# Legit
predict_naive_bayes('Hi mom how are you')

0.12554358867164464

In [15]:
# Legit, but with unknown words
predict_naive_bayes('Hi MOM how aRe yoU afdjsaklfsdhgjasdhfjklsd')

0.12554358867164464

In [16]:
# Legit
predict_naive_bayes('meet me at the lobby of the hotel at nine am')

6.964603508395964e-05

In [17]:
# Spammy
predict_naive_bayes('enter the lottery to win three million dollars')

0.9995234218677429

In [18]:
#Spammy
predict_naive_bayes('buy cheap lottery easy money now')

0.999973472265966

In [19]:
# Legit (hopefully?) :)
predict_naive_bayes('Grokking Machine Learning by Luis Serrano')

0.4197107645484568

In [20]:
# Noise (which the model takes as an empty message).
predict_naive_bayes('asdfgh')

0.2388268156424581

In [21]:
# For the empty message, the probability is simply the prior
# (i.e., the number of spam emails divided by the total number of emails in the dataset)
predict_naive_bayes('')

0.2388268156424581

### Worked examples from the chapter

These cells reproduce the hand calculations in the chapter. The expected outputs are in the comments.

In [22]:
# Medical example: 1,000,000 people, 1 in 10,000 sick, test 99% accurate
sick = 1_000_000 / 10_000
healthy = 1_000_000 - sick
sick_positive = 0.99 * sick        # 99
healthy_positive = 0.01 * healthy  # 9,999
print(round(sick_positive / (sick_positive + healthy_positive), 4))  # 0.0098

0.0098


In [23]:
p_spam, p_ham = 1/5, 4/5
p_lottery_spam, p_lottery_ham = 3/4, 1/16   # lottery: 15 of 20 spam, 5 of 80 ham
p_sale_spam, p_sale_ham = 6/20, 4/80        # sale: 6 of 20 spam, 4 of 80 ham
p_mom_spam, p_mom_ham = 1/20, 10/80         # mom: 1 of 20 spam, 10 of 80 ham

def posterior(spam_likelihoods, ham_likelihoods):
    spam = p_spam * np.prod(spam_likelihoods)
    ham = p_ham * np.prod(ham_likelihoods)
    return round(float(spam / (spam + ham)), 4)

print(posterior([p_lottery_spam], [p_lottery_ham]))                                # 0.75
print(posterior([p_sale_spam], [p_sale_ham]))                                      # 0.6
print(posterior([p_lottery_spam, p_sale_spam], [p_lottery_ham, p_sale_ham]))      # 0.9474
print(posterior([p_lottery_spam, p_sale_spam, p_mom_spam],
                [p_lottery_ham, p_sale_ham, p_mom_ham]))                           # 0.878

0.75
0.6
0.9474
0.878
